# Price Of Ethereum — quickstart

Measure real onchain depth from your own Fynd, understand what a measurement
contains, then record a short history.

**Prerequisites:** a local Fynd (`poe init-worker-pools`, then `fynd serve` from
the same directory) and `TYCHO_API_KEY` in the environment. Install with the viz
extra: `pip install "price-of-ethereum[viz]"`.

Every number below is a Fynd quote or a documented function of quotes. No
oracles, no estimates.

## 1. Connect

Tycho is the only metadata source — decimals, symbol, quality tier, transfer
tax. There is no RPC client anywhere in this package.

In [ ]:
import os

import pandas as pd

from price_of_ethereum import FyndClient, TychoClient, resolve_tokens

# Pick one. The active block must match the chain `fynd serve` is running, and
# SEARCH_MIN/SEARCH_MAX are in whole numeraire units — so they scale with the
# numeraire, not with dollars.

# Ethereum mainnet, ETH priced in USDC. `fynd serve`
TOKEN = "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"        # WETH
NUMERAIRE = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"    # USDC
PAIR = "ETH/USDC"
TYCHO_URL = "https://tycho-beta.propellerheads.xyz"
SEARCH_MIN, SEARCH_MAX = 50.0, 5_000_000.0

# BNB Smart Chain, BTC priced in BNB. `fynd serve --chain bsc`
# TOKEN = "0x7130d2A12B9BCbFAe4f2634d864A1Ee1Ce3Ead9c"      # BTCB
# NUMERAIRE = "0xbb4CdB9CBd36B01bD1cBaEBF2De08d9173bc095c"  # WBNB
# PAIR = "BTCB/WBNB"
# TYCHO_URL = "https://tycho-bsc-beta.propellerheads.xyz"
# SEARCH_MIN, SEARCH_MAX = 0.05, 5_000.0

# Arbitrum, PENDLE priced in ETH. `fynd serve --chain arbitrum`
# TOKEN = "0x0c880f6761F1af8d9Aa9C466984b80DAb9a8c9e8"      # PENDLE
# NUMERAIRE = "0x82aF49447D8a07e3bd95BD0d56f35241523fBab1"  # WETH
# PAIR = "PENDLE/WETH"
# TYCHO_URL = "https://tycho-arbitrum-beta.propellerheads.xyz"
# SEARCH_MIN, SEARCH_MAX = 0.02, 2_000.0

fynd = FyndClient("http://127.0.0.1:3000")
fynd.wait_until_ready()  # cold-start hydration takes 1-5 minutes
chain_id = fynd.info().chain_id

tycho = TychoClient(TYCHO_URL, os.environ["TYCHO_API_KEY"])
tokens = resolve_tokens(tycho, [TOKEN, NUMERAIRE])
token_meta, numeraire_meta = tokens[TOKEN.lower()], tokens[NUMERAIRE.lower()]
TOKEN_SYMBOL, NUMERAIRE_SYMBOL = token_meta.symbol, numeraire_meta.symbol
chain_id, token_meta, numeraire_meta

## 2. Take one snapshot

One spot probe, then a two-sided log-spaced sweep, then anchored bisections at
the headline impact levels. `mid_source` reports how the mid was won —
`sweep_band` is the healthy path; `probe_fallback` and `spot_degraded` mean thin
liquidity or failing quotes.

`samples_per_side` defaults to 100. It is **not** a free resolution knob: it
decides which rungs land in the robust-mid depth band, so it is part of the
mid's definition. Lowering it here only to explore faster.

In [ ]:
from price_of_ethereum import SnapshotConfig, collect_snapshot

config = SnapshotConfig(
    token=token_meta,
    numeraire=numeraire_meta,
    pair=PAIR,
    chain_id=chain_id,
    samples_per_side=40,
    search_min=SEARCH_MIN,
    search_max=SEARCH_MAX,
)
snapshot = collect_snapshot(fynd, config)
snapshot.to_block_row()

## 3. What a measurement contains

`curve` rows are sweep rungs; `anchor` rows are bisected headline levels solved
with `min_responses=0` so the split-routing solver contributes.

In [ ]:
rows = pd.DataFrame.from_records(snapshot.to_rows())
blocks = pd.DataFrame.from_records([snapshot.to_block_row()])
rows.groupby(["kind", "side"]).size()

In [ ]:
rows[rows["kind"] == "curve"].head(3).T

`impact_pct` is measured against `spot` (the $1,000 probe). `price_impact_bps`
is derived against `robust_mid`. `price_impact_bps_raw` is whatever Fynd
reported — nullable, stored for reference, never used for control flow.

In [ ]:
level_columns = [
    "side",
    "target_impact_pct",
    "size_numeraire",
    "impact_pct",
    "price_impact_bps",
    "price_impact_bps_raw",
    "bound",
    "target_reached",
    "derived_from",
]
rows.loc[rows["kind"] == "anchor", level_columns]

## 4. Charts

These are the same figure builders `poe report` uses, so a notebook draws
exactly what the report draws. Pass the token symbols and the axes carry real
units; leave them out and they read `token` / `numeraire`. Prices are always
numeraire per token.

In [ ]:
from price_of_ethereum.dashboard import book_map_figure, cost_curve_figure, spread_curve_figure

cost_curve_figure(rows, numeraire_symbol=NUMERAIRE_SYMBOL).show()

Diamonds are anchored measurements; the line is the bulk sweep. The shaded band
in the book map below is the depth range the robust mid is voted from.

In [ ]:
book_map_figure(rows, snapshot.robust_mid, token_symbol=TOKEN_SYMBOL, numeraire_symbol=NUMERAIRE_SYMBOL).show()
spread_curve_figure(rows, snapshot.robust_mid, numeraire_symbol=NUMERAIRE_SYMBOL).show()

## 5. Where impact stops being monotonic

Measured impact does not have to rise with size: as size grows the router can
recompose the route across more pools and impact can dip. Each negative
difference below is a real routing change, not noise. (Against a single-pool
simulator this comes back empty; against real liquidity it usually does not.)

In [ ]:
curve_buy = rows[(rows["kind"] == "curve") & (rows["side"] == "buy")]
buy = curve_buy.sort_values("size_numeraire").copy()
buy["impact_delta"] = buy["impact_pct"].diff()
buy.loc[
    buy["impact_delta"] < 0,
    ["size_numeraire", "impact_pct", "impact_delta", "n_pools", "route_hash"],
]

## 6. Record a history

`collect_blocks` detects a new block by comparing each snapshot's majority block
to the last recorded one. Rows are appended before the block summary, so the
blocks file is the index of blocks whose rows are fully on disk — join against
it.

In [ ]:
from price_of_ethereum.collect import collect_blocks

result = collect_blocks(fynd, config, out_dir="data", blocks=10)
result

In [ ]:
from price_of_ethereum import load_jsonl, load_parquet, to_parquet
from price_of_ethereum.dashboard import history_health_figure, history_mid_figure

recorded = load_jsonl(result.blocks_path)
summary_columns = [
    "block_number",
    "spot",
    "robust_mid",
    "median_depth",
    "mid_source",
    "mixed_block",
    "duration_ms",
]
recorded[summary_columns]

In [ ]:
history_mid_figure(recorded, token_symbol=TOKEN_SYMBOL, numeraire_symbol=NUMERAIRE_SYMBOL).show()
history_health_figure(recorded).show()  # bars colored where mixed_block fired

JSONL is the collection format; parquet is the analysis format.

In [ ]:
all_rows = load_jsonl(result.rows_path)
to_parquet(all_rows, "data/eth-usdc_1.rows.parquet")
load_parquet("data/eth-usdc_1.rows.parquet").groupby(["block_number", "kind"]).size().head()

## 7. Or write a report

To share a result, or to keep measuring without re-running this notebook, run
the collector and render what it has recorded:

```bash
poe collect --out data &                      # keep measuring
poe report --out data --output report.html    # re-run to refresh the file
```

The report shows the cost curve, book map, round-trip spread, the anchored
levels as a table, and mid / depth / latency across every recorded block. Plotly
is inlined, so the file opens straight from disk and requests nothing
off-machine. It reads only what is on disk and contacts neither Fynd nor Tycho.